# Cluster Sale Detection

Scan clustered wallets for outbound SUI transfers, focusing on labeled CEX addresses to approximate on-chain sales.


In [38]:
from __future__ import annotations

import csv
import json
import requests
from pathlib import Path
from functools import lru_cache
from typing import Dict, List, Set
from datetime import datetime, timezone

try:
    import pandas as pd  # type: ignore
except ImportError:
    pd = None

def resolve_data_dir() -> Path:
    search = [Path.cwd()] + list(Path.cwd().parents)
    for base in search:
        if (base / 'sui_insiders' / 'data').exists():
            return (base / 'sui_insiders' / 'data').resolve()
        if base.name == 'sui_insiders' and (base / 'data').exists():
            return (base / 'data').resolve()
        if (base / 'data').exists() and (base / 'scripts').exists():
            return (base / 'data').resolve()
    return (Path.cwd() / 'data').resolve()

DATA_DIR = resolve_data_dir()
print(f'Data directory: {DATA_DIR}')

CLUSTER_JSON_PATH = DATA_DIR / 'cluster_results.json'
if not CLUSTER_JSON_PATH.exists():
    raise FileNotFoundError(f'Missing cluster JSON: {CLUSTER_JSON_PATH}')

with CLUSTER_JSON_PATH.open('r', encoding='utf-8') as fh:
    CLUSTER_RESULTS: Dict[str, Dict[str, object]] = json.load(fh)

print(f'Loaded {len(CLUSTER_RESULTS)} clusters.')


Data directory: /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data
Loaded 165 clusters.


In [39]:
CEX_WALLETS: Dict[str, List[str]] = {
    'Binance': [
        '0x935029ca5219502a47ac9b69f556ccf6e2198b5e7815cf50f68846f723739cbd',
        '0xac5bceec1b789ff840d7d4e6ce4ce61c90d190a7f8c4f4ddf0bff6ee2413c33c',
    ],
    'Bybit': [
        '0x60dd01bc037e2c1ea2aaf02187701f9f4453ba323338d2f2f521957065b0984d',
    ],
    'Kraken': [
        '0x443cf42b0da43c230bff7a64e69ce25d24d65f49e7c9db6adecc0bd176dba79a',
    ],
    'Bitget': [
        '0xce7e1e38f996cdb2c4b78c1d187d23c1001d7b266f181498677672f9b1e24ea0',
        '0x3a1520d6be3a16f89655f542f6e5b050d3065ff67d7287ed8005259c4a0b36b7',
    ],
    'KuCoin': [
        '0x13c075644dad21bf10cb12447b825b535c24ddae428e27abc5858d1511e24c6d',
        '0xf3bfda04092a842d4c57c61f9d8fb99058792ab2622b2a0c0d83a6ea41de325f',
    ],
    'MEXC': [
        '0xf8b7b95d01ae79756fc3d1bc58675e59a17f07cdafe3d31ea57425adbec2d43f',
    ],
    'Gate': [
        '0x62f36b79d7ea8ae189491854edd9318b29c75346792177b230a95f333ffa53ad',
    ],
    'Crypto.com': [
        '0x7fa064f54d85caed3db8bdb5a3cb5d50b22a7f02f4954059df0e61f87e2bb7e9',
    ],
    'HTX': [
        '0x1f7b27844f2c4a0262b2c481f7ab956d10ace524c5a7b06c3742cfb8701db714',
    ],
    'Coinlist': [
        '0x7fb55e3e2f9ba133efa6223b3c66d15d4b979570c63cff6453041b45e8d6b1e9',
    ],
    'LBank': [
        '0x6cb04412036e910e5fc2237c69d154bb0f69e48a75b030f6919f044f485c8307',
    ],
    'Coinbase': [
        '0x0350925241ae8d4083fe4b1104e14dd28d2ffeced9d7575551c50c41b99995fb',
        '0x2e4706e435195185d69597fdb69bcab9e57a0fa297a14bbfbe70196017de12be',
    ],
    'OKX': [
        '0xab73ad38c63f83eda02182422b545395be1d3caeb54b5869159a9f70b678cd56',
    ],
    'Bitfinex_or_Gate': [
        '0x432875d9abcd8d3ed46e344b7dbeae952d014e8c26fc650e7503417a292cf43d',
    ],
}

ALL_CEX_ADDRESSES: Set[str] = set()
for addrs in CEX_WALLETS.values():
    ALL_CEX_ADDRESSES.update(addrs)
print(f'Total labelled CEX addresses: {len(ALL_CEX_ADDRESSES)}')


Total labelled CEX addresses: 18


In [40]:
DEX_PACKAGES = {
    'bluefin_spot': '0x3492c874c1e3b3e2984e8c41b589e642d4d0a5d6459e5a9cfc2d52fd7c89c267',
    'momentum': '0x70285592c97965e811e0c6f98dccc3a9c2b4ad854b3594faab9597ada267b860',
    'cetus': '0x1eabed72c53feb3805120a081dc15963c204dc8d091542592abaf7a35689b2fb',
}
print(f'DEX packages loaded: {len(DEX_PACKAGES)}')


DEX packages loaded: 3


In [41]:
RPC_URL = 'https://fullnode.mainnet.sui.io:443'
REQUEST_TIMEOUT = 10
MAX_RETRIES = 3

@lru_cache(maxsize=None)
def fetch_transaction_block(tx_digest: str) -> dict | None:
    payload = {
        'jsonrpc': '2.0',
        'id': 1,
        'method': 'sui_getTransactionBlock',
        'params': [tx_digest, {
            'showEvents': True,
            'showInput': True,
        }]
    }
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(RPC_URL, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            body = response.json()
        except Exception as exc:
            last_error = exc
            continue
        if 'error' in body:
            last_error = body['error']
            continue
        return body.get('result')
    print(f'Failed to fetch transaction {tx_digest}: {last_error}')
    return None

@lru_cache(maxsize=None)
def classify_dex(tx_digest: str) -> str | None:
    result = fetch_transaction_block(tx_digest)
    if not result:
        return None
    events = result.get('events') or []
    for event in events:
        event_type = event.get('type', '')
        for label, package in DEX_PACKAGES.items():
            if event_type.startswith(package):
                return label
    transaction = result.get('transaction', {}).get('data', {}).get('transaction')
    if transaction and isinstance(transaction, dict):
        if transaction.get('kind') == 'ProgrammableTransaction':
            for command in transaction.get('commands', []):
                move_call = command.get('MoveCall') if isinstance(command, dict) else None
                if move_call:
                    package = move_call.get('package')
                    for label, pkg in DEX_PACKAGES.items():
                        if package == pkg:
                            return label
    return None


In [42]:
if pd is None:
    raise ImportError('pandas is required for this analysis')

records = []
for csv_path in sorted(DATA_DIR.glob('*.csv')):
    if csv_path.name in {
        'high_inflow_transfers.csv',
        'cluster_results.csv',
        'cluster_cex_transfers.csv',
        'cluster_sales_detail.csv',
        'cluster_sales_summary.csv',
    }:
        continue
    with csv_path.open('r', encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            row['source_csv'] = csv_path.name
            records.append(row)

TRANSFERS_DF = pd.DataFrame(records)
TRANSFERS_DF['amount_sui'] = TRANSFERS_DF['amount_sui'].astype(float)
TRANSFERS_DF['timestamp_ms'] = TRANSFERS_DF['timestamp_ms'].astype(int)
TRANSFERS_DF['timestamp'] = pd.to_datetime(TRANSFERS_DF['timestamp_ms'], unit='ms', utc=True)
print(f'Total transfer rows: {len(TRANSFERS_DF)}')
TRANSFERS_DF.head()


Total transfer rows: 19041


,event_label,tx_digest,checkpoint,timestamp_ms,sender_address,recipient_address,amount_mist,amount_sui,source_csv,timestamp
0,2024-05-02 Unlock,3nXv9CL7nLJWz8Yu8D7nTDTX1PXRFhqzDzu3k9oAxXGz,32961657,1714627200231,0x6f7e38e9d572ba8d92e7f2c1d29653ec5c371835e07c...,0x6f7e38e9d572ba8d92e7f2c1d29653ec5c371835e07c...,8618520,0.008619,2024_05_02_unlock.csv,2024-05-02 05:20:00.231000+00:00
1,2024-05-02 Unlock,13PGbRsszUUdbS8FZiVGjp4eprTn9Mtcy9SkPmzCEyvs,32961657,1714627200231,0x4a768698ed0154897dd68e1ea4fc3c7911776a5a2270...,0x4a768698ed0154897dd68e1ea4fc3c7911776a5a2270...,3925520,0.003926,2024_05_02_unlock.csv,2024-05-02 05:20:00.231000+00:00
2,2024-05-02 Unlock,DNTm2i9MWjf7iGh9y9sdTctgw5XsoTXMWVrpkfkb54Fq,32961658,1714627201248,0xf0f13f7ef773c6246e87a8f059a684d60773f85e992e...,0xf0f13f7ef773c6246e87a8f059a684d60773f85e992e...,3875588,0.003876,2024_05_02_unlock.csv,2024-05-02 05:20:01.248000+00:00
3,2024-05-02 Unlock,mfzeik6sLVrEGnB6AukDyp5u8eSVxE1NZjhcYwRJ9VX,32961659,1714627202268,0xd175da7112f22ff39f2a5866d7c5c5bc09fdb57d8614...,0xd175da7112f22ff39f2a5866d7c5c5bc09fdb57d8614...,3904456,0.003904,2024_05_02_unlock.csv,2024-05-02 05:20:02.268000+00:00
4,2024-05-02 Unlock,Ao7Y715VuW5uV2E4bP27QdcEDFLWsNgMNjzF3SMDKusC,32961659,1714627202268,0xf821d3483fc7725ebafaa5a3d12373d49901bdfce148...,0xf821d3483fc7725ebafaa5a3d12373d49901bdfce148...,3904848,0.003905,2024_05_02_unlock.csv,2024-05-02 05:20:02.268000+00:00


In [43]:
cluster_members: Dict[str, Set[str]] = {}
for seed, summary in CLUSTER_RESULTS.items():
    wallets = set(summary.get('wallets', []))
    if not wallets:
        wallets = {seed}
    cluster_members[seed] = wallets

sender_mask = TRANSFERS_DF['sender_address'].notna() & (TRANSFERS_DF['sender_address'] != '')
OUTBOUND_DF = TRANSFERS_DF[sender_mask].copy()
print(f'Transfers with explicit senders: {len(OUTBOUND_DF)}')


Transfers with explicit senders: 19041


In [44]:
label_lookup = {}
for label, addrs in CEX_WALLETS.items():
    for addr in addrs:
        label_lookup[addr] = label

sales_records: List[dict] = []
for seed_wallet, members in cluster_members.items():
    cluster_txs = OUTBOUND_DF[OUTBOUND_DF['sender_address'].isin(members)]
    for _, row in cluster_txs.iterrows():
        tx_digest = row['tx_digest']
        recipient = row['recipient_address']
        sale_type = None
        target_label = None
        if recipient in label_lookup:
            sale_type = 'CEX'
            target_label = label_lookup[recipient]
        else:
            dex_label = classify_dex(tx_digest)
            if dex_label:
                sale_type = 'DEX'
                target_label = dex_label
        if sale_type:
            sales_records.append({
                'seed_wallet': seed_wallet,
                'sale_type': sale_type,
                'target_label': target_label,
                'tx_digest': tx_digest,
                'sender_address': row['sender_address'],
                'recipient_address': recipient,
                'amount_sui': row['amount_sui'],
                'timestamp': row['timestamp'],
                'event_label': row['event_label'],
                'source_csv': row['source_csv'],
            })
SALES_DF = pd.DataFrame(sales_records)
print(f'Sale rows detected: {len(SALES_DF)}')
if not SALES_DF.empty:
    display(SALES_DF.head())


Sale rows detected: 1470


,seed_wallet,sale_type,target_label,tx_digest,sender_address,recipient_address,amount_sui,timestamp,event_label,source_csv
0,0x3444b3ee072e398e5db552ef69bdaf5ebb4b7500b908...,CEX,OKX,F9k65hbGazGMxceAKQgCsVgj6KbLraZKodRysv25bDnv,0xa630dd7c337cdde2b2022a3e0bb028c0513a3b65fd6d...,0xab73ad38c63f83eda02182422b545395be1d3caeb54b...,298.297024,2024-05-02 05:22:47.254000+00:00,2024-05-02 Unlock,2024_05_02_unlock.csv
1,0x3444b3ee072e398e5db552ef69bdaf5ebb4b7500b908...,CEX,OKX,9ikgQcmNqyEUWS63etyk5v3VwvsvHjGDXQUbgKShWLBJ,0x55f4abea6c99f1605f403f1e3e334746eacb06994b4d...,0xab73ad38c63f83eda02182422b545395be1d3caeb54b...,6756.519521,2024-05-02 05:25:47.947000+00:00,2024-05-02 Unlock,2024_05_02_unlock.csv
2,0x3444b3ee072e398e5db552ef69bdaf5ebb4b7500b908...,CEX,KuCoin,4ckw4GbLHbZTbKen8d7fanLTb9YhkdGt8qdouXeA61pj,0xe39d8ef9580dc417ba0e8386ca871cd3051da5339784...,0x13c075644dad21bf10cb12447b825b535c24ddae428e...,131.332133,2024-05-02 05:30:03.352000+00:00,2024-05-02 Unlock,2024_05_02_unlock.csv
3,0x3444b3ee072e398e5db552ef69bdaf5ebb4b7500b908...,CEX,Binance,4pKMZiZYaN1SRvJYTy575oDCrEzGPPecthXq1ocmigB7,0xd13533955cb198a0c580d3296b3085c816edeffa1aca...,0x935029ca5219502a47ac9b69f556ccf6e2198b5e7815...,11192.940208,2024-05-02 05:33:02.092000+00:00,2024-05-02 Unlock,2024_05_02_unlock.csv
4,0x3444b3ee072e398e5db552ef69bdaf5ebb4b7500b908...,CEX,OKX,2xAweV42dSHWmsaAyAHb7gRvLiwFsr2mTUkkGyC9kbH9,0x641c3732c875b0a0f53cf22cb394eebfdb383c89fd9a...,0xab73ad38c63f83eda02182422b545395be1d3caeb54b...,426.997024,2024-05-02 05:33:47.066000+00:00,2024-05-02 Unlock,2024_05_02_unlock.csv


In [45]:
if not SALES_DF.empty:
    summary = (
        SALES_DF.groupby('seed_wallet')
        .agg(
            first_sale=('timestamp', 'min'),
            last_sale=('timestamp', 'max'),
            total_sui_sold=('amount_sui', 'sum'),
            sale_count=('tx_digest', 'nunique')
        )
        .reset_index()
        .sort_values('total_sui_sold', ascending=False)
    )
    display(summary)
else:
    summary = pd.DataFrame(columns=['seed_wallet', 'first_sale', 'last_sale', 'total_sui_sold', 'sale_count'])


,seed_wallet,first_sale,last_sale,total_sui_sold,sale_count
39,0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc0...,2024-06-01 15:50:17.412000+00:00,2024-08-31 23:35:08.126000+00:00,3.077466e+06,32
93,0xecad8b6df613da99ebb12a2d42dffc61503628f0170d...,2024-06-01 15:50:17.412000+00:00,2024-08-31 23:35:08.126000+00:00,3.077466e+06,32
79,0xcf30da961e1d310546f4df6dcbac0ace091852c77ed2...,2024-06-01 15:50:17.412000+00:00,2024-08-31 23:35:08.126000+00:00,3.077466e+06,32
12,0x1c9deef7e10b88cca4211e5d688bb496d6843d916948...,2024-08-01 12:54:08.220000+00:00,2024-08-31 23:39:27.942000+00:00,1.683524e+06,7
18,0x258dc8a75c7d276134150a25ec70cee986742de5de97...,2024-07-02 02:20:07.335000+00:00,2024-08-01 13:08:17.621000+00:00,7.007741e+05,7
...,...,...,...,...,...
19,0x2774ac65ec2cb23e07c295be84bdd904a71e1244ca54...,2024-06-01 15:50:17.412000+00:00,2024-06-01 16:09:34.332000+00:00,2.138587e+01,4
24,0x311a09b621330bdffa4294d5f893ff605dcef04b6051...,2024-06-01 15:50:17.412000+00:00,2024-06-01 16:09:34.332000+00:00,2.138587e+01,4
61,0x9ac78c4a91895b47d22be1d710807d3833b75779d99a...,2024-06-01 15:56:27.517000+00:00,2024-06-01 16:09:34.332000+00:00,1.712272e+01,3
41,0x6cd00b0ccb13fe511776c8936af4ae74360de2f475d6...,2024-05-02 05:27:54.034000+00:00,2024-05-02 05:27:54.034000+00:00,1.303798e+01,1


In [46]:
if not SALES_DF.empty:
    print(f'Total sale transactions detected: {len(SALES_DF)}')
else:
    print('Total sale transactions detected: 0')


Total sale transactions detected: 1470


In [47]:
if not SALES_DF.empty:
    detail_path = DATA_DIR / 'cluster_sales_detail.csv'
    summary_path = DATA_DIR / 'cluster_sales_summary.csv'
    SALES_DF.to_csv(detail_path, index=False)
    summary.to_csv(summary_path, index=False)
    print(f'Saved detailed sales to {detail_path}')
    print(f'Saved summary to {summary_path}')
else:
    print('No sales detected; nothing exported.')


Saved detailed sales to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/cluster_sales_detail.csv
Saved summary to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/cluster_sales_summary.csv


## Next Extensions

- Add heuristics for on-chain swap destinations (DEX pools, bridges) to capture non-CEX selling activity.
- Enrich with price data to estimate USD value of each sale.
- Integrate attribution tiers (team, investors) once clusters are labeled.
